In [14]:
from collections import deque
from itertools import combinations
import os

## Аудиториска вежба со поставување на N кралици на NxN шаховска табла

In [24]:
def is_valid(new_state, new_queen_i, new_queen_j):
    vertical_check  = new_queen_j not in new_state
    if not vertical_check :
        return False
    main_diagonal = new_queen_i - new_queen_j
    anti_diagonal = new_queen_j + new_queen_i
    other_queens = new_state[ : N - new_state.count(None)]
    for other_queen_i, other_queen_j  in enumerate(other_queens):
        if other_queen_i - other_queen_j == main_diagonal:
            return False
        if other_queen_i + other_queen_j == anti_diagonal:
            return False
    return True

In [25]:
def expand_state(state):
    new_states = []
    available_places = state.count(None)
    if not available_places:
        return []
    new_queen_i = N - available_places
    for new_queen_j in range (N):
        if is_valid(state, new_queen_i, new_queen_j):
            new_state = list(state)
            new_state[new_queen_i] = new_queen_j
            new_states.append(tuple(new_state))
    return new_states

In [26]:
def search(initial_state, alg):
    visited = {initial_state}
    states_queue = deque([initial_state])
    while states_queue:
        state_to_expand = states_queue.popleft()
        for next_state in expand_state(state_to_expand):
            if next_state not in visited:
                if next_state.count(None) == 0:
                    return next_state
                visited.add(next_state)
                if alg == 'dfs':
                    states_queue.appendleft(next_state)
                elif alg == 'bfs':
                    states_queue.append(next_state)

In [27]:
def visualise_queens(queens):
    import numpy as np
    import skimage
    from skimage import io

    if not queens:
        print('Не постои реше.astype(np.uint8)ние за N =', N)
        return
    border_color = (0, 0, 0, 1)
    queen = skimage.img_as_float(io.imread('images/queen.png'))
    table = np.zeros((queen.shape[0] * N, queen.shape[1] * N, queen.shape[2]))
    margin = queen.shape[0] // 20
    for i, j in enumerate(queens):
        table[i*queen.shape[0]:(i+1)*queen.shape[0], j*queen.shape[1]:(j+1)*queen.shape[1]] = queen
    for index in range(1, N):
        table[index * queen.shape[0] - margin: index * queen.shape[0] + margin] = border_color
        table[:, index * queen.shape[1] - margin: index * queen.shape[1] + margin] = border_color
    image_directory = 'queens'
    os.makedirs(f'images/{image_directory}', exist_ok=True)
    io.imsave(f'images/{image_directory}/{N}.png', 255*table.astype(np.uint8))
    return f'Погледни ја сликата images/{image_directory}/{N}.png'

In [28]:
N = 8
initial_state = (None,) * N
queens = search(initial_state, alg='dfs')
queens

(7, 3, 0, 2, 5, 1, 6, 4)

In [30]:
%%time

N = 12
initial_state = (None,) * N
queens = search(initial_state, alg='dfs')
queens

CPU times: user 1.41 ms, sys: 0 ns, total: 1.41 ms
Wall time: 1.41 ms


(11, 9, 7, 4, 2, 0, 6, 1, 10, 5, 3, 8)

## I колоквиум 2025

#### На табла NxN да се постави максимален број на коњи, без притоа истите да се напаѓаат.

In [69]:
N = 8

def is_valid(state, new_r, new_c):
    """ Проверува дали е безбедно да се постави коњ на позиција (new_r, new_c) """
    # Сите 8 можни потези на коњот (буквата Г)
    moves = [
        (-2, -1), (-2, 1), 
        (-1, -2), (-1, 2), 
        (1, -2),  (1, 2), 
        (2, -1),  (2, 1)
    ]
    
    for dr, dc in moves:
        r, c = new_r + dr, new_c + dc
        # Проверка дали позицијата е во рамките на таблата
        if 0 <= r < N and 0 <= c < N:
            # Ако на таа позиција веќе има друг коњ (1), тогаш не е валидно
            if state[r][c] == 1:
                return False
    return True

In [70]:
def expand_state(state):
    new_states = []
    empty_pos = find_next_empty(state)

    if empty_pos is None:
        return []
    r,c = empty_pos

    if is_valid(state,r,c):
        temp_board = [list(row) for row in state]
        temp_board[r][c] = 1
        new_state_with_knight = tuple(tuple(row) for row in temp_board)
        new_states.append(new_state_with_knight)
        
    else:
    
        temp_board = [list(row) for row in state]
        temp_board[r][c] = 2
        new_state_with_knight = tuple(tuple(row) for row in temp_board)
        new_states.append(new_state_with_knight)
    return new_states



In [71]:
def count_knights(state):
    """ Го пребројува вкупниот број на поставени коњи (единици) на таблата """
    return sum(row.count(1) for row in state)

In [79]:
def find_next_empty(state):
    for r in range (N):
        for c in range (N):
            if state[r][c] == 0 and (r + c) % 2 == 0:
                return r, c
                
    for r in range(N):
        for c in range(N):
            if state[r][c] == 0 and (r + c) % 2 != 0:
                return r, c
                
    return None

In [80]:
def search(initial_state):
    visited = {initial_state}
    states_queue = deque([initial_state])
    max_knights = 0
    while states_queue:
        state_to_expand = states_queue.popleft()
        for next_state in expand_state(state_to_expand):
            if next_state not in visited:
                visited.add(next_state)
                current_max = count_knights(next_state)
                if current_max>max_knights:
                    max_knights = current_max
                    max_state = next_state
                states_queue.appendleft(next_state)
    return max_state, max_knights

In [81]:
initial_state = tuple(tuple( 0 for x in range (N)) for x in range(N))
initial_state

((0, 0, 0, 0, 0, 0, 0, 0),
 (0, 0, 0, 0, 0, 0, 0, 0),
 (0, 0, 0, 0, 0, 0, 0, 0),
 (0, 0, 0, 0, 0, 0, 0, 0),
 (0, 0, 0, 0, 0, 0, 0, 0),
 (0, 0, 0, 0, 0, 0, 0, 0),
 (0, 0, 0, 0, 0, 0, 0, 0),
 (0, 0, 0, 0, 0, 0, 0, 0))

In [82]:
result, max_knights = search(initial_state)
max_knights

32

### Функции за пресметка на растојание: 
##### - Ако е произволно движење на агентот се користи Евклидово растојание
##### - Ако се движи во 4 насоки се користи Менхетен растојание, ако се движи во 8 насоки се користи Чебишев.

### А* пребарување
#### Ако се отстрани цената на патеката алгоритамот работи како алчен алгоритам. Ако се отстрани дојавата работи како алгоритам со униформна цена. Ако се постави цената кај пребарување по униформна цена за сите на 1, работи како пребарување по широчина BFS.

### Испитна јануари 2025
##### Агент се движи низ матрица каде на полињата има поставени различни фигури. Целта е агентот да собери 3 кругчиња.
##### Пошто не знам како била кај Стефан поставена, си напрајв произволна матрица В и претпоставив дека кругчиња има на позициите со 1.

In [1]:
def check_goal(state):
    x,y,collected = state
    if len(collected)== 3:
        return True
    return False

In [3]:
B = [[1,2,1,2,3],
     [3,2,3,1,2]]

ROWS = 2
COLUMNS = 5

In [17]:
def is_valid_square(x,y):
    if not (0 <= x < ROWS) or not (0 <= y < COLUMNS):
        return False        
    return True

is_valid_square(0, 0)

True

In [18]:
def expand_state_4(state):
    next_states = []
    r, c, visited_circles = state  # visited_circles е торка од координати, на пример: ((0,2), (1,3))
    
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for dr, dc in directions:
        nr, nc = r + dr, c + dc
        if is_valid_square(nr, nc):
            if B[nr][nc] == 1 and (nr, nc) not in visited_circles:
                new_visited = visited_circles + ((nr, nc),)
                next_states.append((nr, nc, new_visited))
            else:
                next_states.append((nr, nc, visited_circles))
                
    return next_states

In [24]:
def search(initial_state, alg):
    visited = {initial_state}
    states_queue = deque([initial_state])
    while states_queue:
        state_to_expand = states_queue.popleft()
        if check_goal(state_to_expand):
            return f"Целта е постигната! Крајна состојба: {state_to_expand}"
        for next_state in expand_state_4(state_to_expand):
            if next_state not in visited:
                visited.add(next_state)
                if alg == 'dfs':
                    states_queue.appendleft(next_state)
                elif alg == 'bfs':
                    states_queue.append(next_state)
    return "Нема решение"

In [25]:
x,y = 0,0
initial_state = (x,y,())
type(initial_state)

tuple

In [30]:
%%time
search(initial_state,'bfs')

CPU times: user 76 μs, sys: 11 μs, total: 87 μs
Wall time: 89.2 μs


'Целта е постигната! Крајна состојба: (1, 3, ((0, 0), (0, 2), (1, 3)))'

In [31]:
%%time
search(initial_state,'dfs')

CPU times: user 30 μs, sys: 4 μs, total: 34 μs
Wall time: 35.8 μs


'Целта е постигната! Крајна состојба: (0, 0, ((0, 2), (1, 3), (0, 0)))'

##### Заклучок: Ако е неинформирано пребарување и по широчина и по длабочина се добива слично време. Можно е во испитот да била поинаку поставена задачата и да имало други услови, тогаш место bfs/dfs би се искористил друг алгоритам. Логиката за expand_state е иста.